In [24]:
!pip install sentence-transformers pandas numpy scikit-learn

In [25]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances, manhattan_distances

In [26]:
df = pd.read_csv("cleaned_transcripts.csv")

print("Dataset loaded successfully")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully
Shape: (398, 4)


,video_id,title,datetime,transcript
0,E-CH3-VyVck,Why is Git INSANELY Fast? (And How Commits Ac...,2026-02-24 11:00:29+00:00,You have typed get commit thousands of times. ...
1,URI5GsOBznk,YouTube Recommendation Engine: Complete Meltdo...,2026-02-21 04:26:16+00:00,"YouTube went down. [music] 350,000 users repor..."
2,8d2eG7bdepQ,Implementing OAuth and MFA: Full Authenticatio...,2026-02-18 11:00:16+00:00,Every time you click sign in with Google or co...
3,GQ6piqfwr5c,"How Stripe Built AI Agents That Write 1,000+ P...",2026-02-14 13:02:10+00:00,Scribe just revealed something big. They have ...
4,p5hA8rpCRXw,The Selenium Problem: Why QA Teams Waste 40% o...,2026-02-11 11:01:25+00:00,Here's a stat that surprised me. QA team spend...


In [27]:
model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

print("Loaded model:", model_name)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9544.72it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: all-MiniLM-L6-v2


In [28]:
model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

print("Loaded model:", model_name)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11385.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: all-MiniLM-L6-v2


In [29]:
titles = df["title"].fillna("").astype(str).tolist()
transcripts = df["transcript"].fillna("").astype(str).tolist()

title_embeddings = model.encode(titles, show_progress_bar=True)
transcript_embeddings = model.encode(transcripts, show_progress_bar=True)

print("Title embeddings shape:", title_embeddings.shape)
print("Transcript embeddings shape:", transcript_embeddings.shape)

Batches: 100%|██████████| 13/13 [00:09<00:00,  1.43it/s]

Title embeddings shape: (398, 384)
Transcript embeddings shape: (398, 384)


In [30]:
def compute_scores(query_embedding, metric_name="cosine"):
    query_embedding = query_embedding.reshape(1, -1)

    if metric_name == "cosine":
        title_scores = cosine_similarity(query_embedding, title_embeddings)[0]
        transcript_scores = cosine_similarity(query_embedding, transcript_embeddings)[0]

    elif metric_name == "euclidean":
        title_scores = -euclidean_distances(query_embedding, title_embeddings)[0]
        transcript_scores = -euclidean_distances(query_embedding, transcript_embeddings)[0]

    elif metric_name == "manhattan":
        title_scores = -manhattan_distances(query_embedding, title_embeddings)[0]
        transcript_scores = -manhattan_distances(query_embedding, transcript_embeddings)[0]

    else:
        raise ValueError("Unsupported metric")

    return title_scores, transcript_scores

In [31]:
def returnSearchResults(
    query,
    df,
    metric_name="cosine",
    top_k=5,
    threshold=0.5,
    title_weight=0.3,
    transcript_weight=0.7
):
    # Step 1: encode query
    query_embedding = model.encode([query])[0]

    # Step 2: compute title and transcript scores
    title_scores, transcript_scores = compute_scores(query_embedding, metric_name)

    # Step 3: combine scores
    final_scores = (title_weight * title_scores) + (transcript_weight * transcript_scores)

    # Step 4: attach scores to dataframe
    results_df = df.copy()
    results_df["title_score"] = title_scores
    results_df["transcript_score"] = transcript_scores
    results_df["final_score"] = final_scores

    # Step 5: apply threshold
    filtered_df = results_df[results_df["final_score"] >= threshold]

    # Step 6: sort and get top-k
    filtered_df = filtered_df.sort_values(by="final_score", ascending=False).head(top_k)

    return filtered_df[["video_id", "title", "datetime", "final_score"]]

In [32]:
df = pd.read_csv("cleaned_transcripts.csv")

query = "What is ai agent?"

results = returnSearchResults(
    query=query,
    df=df,
    metric_name="cosine",
    top_k=5,
    threshold=0.5,
    title_weight=0.3,
    transcript_weight=0.7
)

results

,video_id,title,datetime,final_score
83,Jj1-zb38Yfw,Agentic AI Explained So Anyone Can Get It!,2025-06-18 10:00:35+00:00,0.764929
106,MWv33xofJ40,AI Assistant vs AI Agent 🤖 | What&39;s the Dif...,2025-04-25 12:30:21+00:00,0.709019
102,9UNPwGRNo_k,I Tested DeepAgent: How AI Agents Use MCP to T...,2025-04-30 10:00:27+00:00,0.672424
17,voSYs_XDsog,I Built a Game Without Writing Code (AI Agent ...,2025-12-10 12:02:04+00:00,0.630302
220,SYHqSAWQ4NY,"Autonomous AI Agents, Large Action Model and M...",2024-11-22 18:02:34+00:00,0.618158


In [33]:
query = "What is ai agent?"

for metric in ["cosine", "euclidean", "manhattan"]:
    print(f"\nMetric: {metric}")
    display(returnSearchResults(
        query=query,
        df=df,
        metric_name=metric,
        top_k=5,
        threshold=0.5,
        title_weight=0.3,
        transcript_weight=0.7
    ))


Metric: cosine


,video_id,title,datetime,final_score
83,Jj1-zb38Yfw,Agentic AI Explained So Anyone Can Get It!,2025-06-18 10:00:35+00:00,0.764929
106,MWv33xofJ40,AI Assistant vs AI Agent 🤖 | What&39;s the Dif...,2025-04-25 12:30:21+00:00,0.709019
102,9UNPwGRNo_k,I Tested DeepAgent: How AI Agents Use MCP to T...,2025-04-30 10:00:27+00:00,0.672424
17,voSYs_XDsog,I Built a Game Without Writing Code (AI Agent ...,2025-12-10 12:02:04+00:00,0.630302
220,SYHqSAWQ4NY,"Autonomous AI Agents, Large Action Model and M...",2024-11-22 18:02:34+00:00,0.618158



Metric: euclidean


,video_id,title,datetime,final_score



Metric: manhattan


,video_id,title,datetime,final_score


In [34]:
query = "What is ai agent?"

for threshold in [0.3, 0.5, 0.7]:
    print(f"\nThreshold: {threshold}")
    display(returnSearchResults(
        query=query,
        df=df,
        metric_name="cosine",
        top_k=5,
        threshold=threshold,
        title_weight=0.3,
        transcript_weight=0.7
    ))


Threshold: 0.3


,video_id,title,datetime,final_score
83,Jj1-zb38Yfw,Agentic AI Explained So Anyone Can Get It!,2025-06-18 10:00:35+00:00,0.764929
106,MWv33xofJ40,AI Assistant vs AI Agent 🤖 | What&39;s the Dif...,2025-04-25 12:30:21+00:00,0.709019
102,9UNPwGRNo_k,I Tested DeepAgent: How AI Agents Use MCP to T...,2025-04-30 10:00:27+00:00,0.672424
17,voSYs_XDsog,I Built a Game Without Writing Code (AI Agent ...,2025-12-10 12:02:04+00:00,0.630302
220,SYHqSAWQ4NY,"Autonomous AI Agents, Large Action Model and M...",2024-11-22 18:02:34+00:00,0.618158



Threshold: 0.5


,video_id,title,datetime,final_score
83,Jj1-zb38Yfw,Agentic AI Explained So Anyone Can Get It!,2025-06-18 10:00:35+00:00,0.764929
106,MWv33xofJ40,AI Assistant vs AI Agent 🤖 | What&39;s the Dif...,2025-04-25 12:30:21+00:00,0.709019
102,9UNPwGRNo_k,I Tested DeepAgent: How AI Agents Use MCP to T...,2025-04-30 10:00:27+00:00,0.672424
17,voSYs_XDsog,I Built a Game Without Writing Code (AI Agent ...,2025-12-10 12:02:04+00:00,0.630302
220,SYHqSAWQ4NY,"Autonomous AI Agents, Large Action Model and M...",2024-11-22 18:02:34+00:00,0.618158



Threshold: 0.7


,video_id,title,datetime,final_score
83,Jj1-zb38Yfw,Agentic AI Explained So Anyone Can Get It!,2025-06-18 10:00:35+00:00,0.764929
106,MWv33xofJ40,AI Assistant vs AI Agent 🤖 | What&39;s the Dif...,2025-04-25 12:30:21+00:00,0.709019


In [35]:
queries_df = pd.read_csv("query_video_mapping.csv")
queries_df.head()

,query,relevant_video_id
0,What is diagram visually?,67Ekk-rM5dE
1,Explain performance dns,xv0Be4QfkH0
2,Explain benchmarks dragonfly,j1PkkSddZcE
3,How does processing stream work?,mG3xQb_-rV4
4,How does sql injection work?,WUBWIVCJLHI


In [36]:
evaluation_results = []

for _, row in queries_df.iterrows():
    query = row["query"]
    expected_video = row["relevant_video_id"]

    results = returnSearchResults(
        query=query,
        df=df,
        metric_name="cosine",
        top_k=5,
        threshold=0.5,
        title_weight=0.3,
        transcript_weight=0.7
    )

    retrieved_ids = results["video_id"].tolist()

    if expected_video in retrieved_ids:
        rank = retrieved_ids.index(expected_video) + 1
    else:
        rank = None

    evaluation_results.append({
        "Query": query,
        "Expected Video": expected_video,
        "Retrieved Rank": rank
    })

eval_df = pd.DataFrame(evaluation_results)
eval_df.head(10)

,Query,Expected Video,Retrieved Rank
0,What is diagram visually?,67Ekk-rM5dE,NaN
1,Explain performance dns,xv0Be4QfkH0,NaN
2,Explain benchmarks dragonfly,j1PkkSddZcE,1.0
3,How does processing stream work?,mG3xQb_-rV4,1.0
4,How does sql injection work?,WUBWIVCJLHI,1.0
5,What is architecture sidecar?,FMxUYhYDiys,1.0
6,What is apis harder?,qnXTTmPPiz0,1.0
7,How does advantages fusionaut work?,t3HvCLYnrRY,NaN
8,Explain legacy microservices,DpuQ3-7e-rY,2.0
9,What is structured apps?,t-tTh94NI78,NaN


In [37]:
valid_ranks = eval_df["Retrieved Rank"].dropna()

top1_recall = (valid_ranks <= 1).mean()
top3_recall = (valid_ranks <= 3).mean()
top5_recall = (valid_ranks <= 5).mean()
avg_rank = valid_ranks.mean()

summary_df = pd.DataFrame([{
    "Top-1 Recall": round(top1_recall, 3),
    "Top-3 Recall": round(top3_recall, 3),
    "Top-5 Recall": round(top5_recall, 3),
    "Avg Rank": round(avg_rank, 2)
}])

summary_df

,Top-1 Recall,Top-3 Recall,Top-5 Recall,Avg Rank
0,0.803,1.0,1.0,1.27


In [38]:
eval_df.to_csv("module7_evaluation_output.csv", index=False)
summary_df.to_csv("module7_summary.csv", index=False)

print("Module 7 outputs saved")

Module 7 outputs saved
